<p align="center"><a href="https://www.plus2net.com/python/json-load.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>\n\n# Python json.load(): Parse JSON from Files & Streams\n\n**Companion Tutorial:** [https://www.plus2net.com/python/json-load.php](https://www.plus2net.com/python/json-load.php)\n\nThis companion notebook runs all examples from the `json.load()` tutorial, demonstrating stream deserialization, custom object hooks, Decimal precision, and self-checking assertions.

### 1. Basic File Stream Deserialization\nCreate a simulated file stream and load it into a native Python dictionary.

In [ ]:
import json\nimport io\n\nsample_json = '''{\n  "app_name": "Plus2Net Inventory",\n  "version": 3.1,\n  "active": true,\n  "database": {\n    "host": "localhost",\n    "port": 3306\n  },\n  "supported_formats": ["json", "csv", "xlsx"]\n}'''\n\n# Load from in-memory stream\nstream = io.StringIO(sample_json)\ndata = json.load(stream)\n\nprint("Type:", type(data))\nprint("Application:", data["app_name"])\nprint("DB Port:    ", data["database"]["port"]) 

### 2. Handling JSONDecodeError\nVerify how Python catches invalid syntax like unquoted property names or trailing commas.

In [ ]:
corrupted_json = '{"name": "Alice", trailing_comma: true,}'\ncorrupted_stream = io.StringIO(corrupted_json)\n\ntry:\n    json.load(corrupted_stream)\nexcept json.JSONDecodeError as err:\n    print(f"Caught expected syntax error on line {err.lineno}: {err.msg}")

### 3. Financial Precision with parse_float=Decimal\nPrevent IEEE 754 float rounding inaccuracies by decoding numbers directly into Decimal objects.

In [ ]:
from decimal import Decimal\n\ninvoice_json = '{"item": "GPU", "price": 799.99, "tax": 64.00}'\nstream = io.StringIO(invoice_json)\nrecord = json.load(stream, parse_float=Decimal)\n\nprint("Price:", record["price"], "(type:", type(record["price"]), ")")\nprint("Total:", record["price"] + record["tax"])

### 4. Custom Object Decoder with object_hook\nAutomatically instantiate domain classes from decoded JSON objects.

In [ ]:
class Employee:\n    def __init__(self, name, department):\n        self.name = name\n        self.department = department\n\n    def __repr__(self):\n        return f"Employee(name='{self.name}', dept='{self.department}')"\n\ndef employee_decoder(dct):\n    if "name" in dct and "department" in dct:\n        return Employee(dct["name"], dct["department"])\n    return dct\n\nemp_json = '{"name": "Diana", "department": "DevOps"}'\nemp_obj = json.load(io.StringIO(emp_json), object_hook=employee_decoder)\nprint("Decoded Object:", emp_obj)

### 5. Hands-on Practice Exercise: Configuration Validator\nRun the self-checking unit assertions below to verify complete configuration validation.

In [ ]:
config_payload = '''{\n  "service": "billing_worker",\n  "port": 9000,\n  "max_workers": 8,\n  "debug": false,\n  "allowed_origins": ["https://plus2net.com", "http://localhost:3000"]\n}'''\n\ndef load_and_validate_config(stream):\n    config = json.load(stream)\n    required_keys = ["service", "port", "max_workers"]\n    for key in required_keys:\n        if key not in config:\n            raise ValueError(f"Missing key: {key}")\n    return config\n\nstream_obj = io.StringIO(config_payload)\nparsed_cfg = load_and_validate_config(stream_obj)\n\n# Unit Assertions\nassert parsed_cfg["service"] == "billing_worker"\nassert parsed_cfg["port"] == 9000\nassert parsed_cfg["debug"] is False\nassert len(parsed_cfg["allowed_origins"]) == 2\n\nprint("Validated Config:", parsed_cfg)\nprint("All assertions passed successfully!")